# Analisis Data Mining - Heart Disease
Notebook ini berisi tahapan CRISP-DM, preprocessing, classification, clustering, evaluasi, dan penyimpanan model.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report, silhouette_score, davies_bouldin_score
import joblib

In [ ]:
df = pd.read_csv('../dataset/heart.csv.csv')
df.head()

In [ ]:
df.shape, df.info(), df.isna().sum(), df.duplicated().sum(), df['target'].value_counts()

## Data Understanding
Dataset memiliki 1.025 record dan 14 kolom. Kolom `target` adalah label biner: 0 tidak ada penyakit jantung, 1 ada indikasi penyakit jantung.

In [ ]:
df.describe()

In [ ]:
df['target'].value_counts().sort_index().plot(kind='bar')
plt.title('Distribusi Target')
plt.xlabel('Target')
plt.ylabel('Jumlah')
plt.show()

## Modeling - Classification
Dua algoritma dibandingkan: Logistic Regression dan Random Forest.

In [ ]:
X = df.drop(columns=['target'])
y = df['target']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
models = {
    'Logistic Regression': Pipeline([('scaler', StandardScaler()), ('clf', LogisticRegression(max_iter=1000, random_state=42))]),
    'Random Forest': RandomForestClassifier(n_estimators=300, random_state=42, class_weight='balanced')
}
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    proba = model.predict_proba(X_test)[:, 1]
    print('\n', name)
    print('Accuracy:', accuracy_score(y_test, pred))
    print('Precision:', precision_score(y_test, pred))
    print('Recall:', recall_score(y_test, pred))
    print('F1:', f1_score(y_test, pred))
    print('ROC AUC:', roc_auc_score(y_test, proba))
    print('Confusion Matrix:\n', confusion_matrix(y_test, pred))

## Validasi Tambahan: Data Deduplikasi
Karena terdapat banyak baris duplikat, evaluasi tambahan dilakukan setelah menghapus duplikasi untuk menghindari kesimpulan yang terlalu optimistis.

In [ ]:
df_dedup = df.drop_duplicates()
Xd = df_dedup.drop(columns=['target'])
yd = df_dedup['target']
Xtr, Xte, ytr, yte = train_test_split(Xd, yd, test_size=0.2, random_state=42, stratify=yd)
for name, model in models.items():
    model.fit(Xtr, ytr)
    pred = model.predict(Xte)
    proba = model.predict_proba(Xte)[:, 1]
    print('\n', name)
    print('Accuracy:', accuracy_score(yte, pred))
    print('Precision:', precision_score(yte, pred))
    print('Recall:', recall_score(yte, pred))
    print('F1:', f1_score(yte, pred))
    print('ROC AUC:', roc_auc_score(yte, proba))
    print('Confusion Matrix:\n', confusion_matrix(yte, pred))

## Modeling - Clustering
K-Means digunakan untuk segmentasi profil pasien. Nilai k dipilih berdasarkan silhouette score.

In [ ]:
scaler = StandardScaler()
Xs = scaler.fit_transform(X)
for k in range(2, 7):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(Xs)
    print(k, 'Silhouette:', silhouette_score(Xs, labels), 'DBI:', davies_bouldin_score(Xs, labels))

In [ ]:
final_model = Pipeline([('scaler', StandardScaler()), ('clf', LogisticRegression(max_iter=1000, random_state=42))])
final_model.fit(X, y)
joblib.dump(final_model, '../model/heart_model.pkl')
km = KMeans(n_clusters=2, random_state=42, n_init=10).fit(Xs)
joblib.dump({'scaler': scaler, 'model': km, 'features': list(X.columns)}, '../model/kmeans_model.pkl')